# Amazon Product Review Scraper
## Dataset Collection for Sentiment Analysis Assignment

This notebook scrapes product reviews from Amazon India and stores them in CSV format for analysis.

**Requirements:**
- Minimum 100 reviews
- Clean, readable data
- CSV format with review_text column
- No restricted or sensitive content
- Additional columns: rating, reviewer_name, review_date

**Note:** The browser will open automatically. Keep it in focus while the script runs.

In [1]:
# Section 1: Import Required Libraries
import subprocess
import sys

print("Installing required packages...")
packages = ['selenium', 'beautifulsoup4', 'pandas', 'webdriver-manager']
for package in packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"✓ {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✓ {package} installed")

print("\n✓ All packages installed successfully!\n")

# Import libraries
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
import pandas as pd
import time
from datetime import datetime

print("✓ All libraries imported successfully!")

Installing required packages...
✓ selenium already installed
Installing beautifulsoup4...
✓ beautifulsoup4 installed
✓ pandas already installed
✓ webdriver-manager already installed

✓ All packages installed successfully!

✓ All libraries imported successfully!


In [2]:
# Section 2: Set Up Selenium WebDriver
print("Setting up Selenium WebDriver...")

options = webdriver.ChromeOptions()
options.add_argument('--start-maximized')
options.add_argument('--disable-blink-features=AutomationControlled')
options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
driver.implicitly_wait(10)

print("✓ WebDriver initialized successfully!")

Setting up Selenium WebDriver...
✓ WebDriver initialized successfully!


In [3]:
# Section 3: Navigate to Amazon India and Search Product
print("Navigating to Amazon India...")

driver.get('https://www.amazon.in')
time.sleep(2)

try:
    consent_button = driver.find_element(By.ID, 'sp-cc-accept')
    consent_button.click()
    print("✓ Consent dialog closed")
except:
    print("No consent dialog found")

time.sleep(1)

# Change this to search for different products
SEARCH_PRODUCT = "OnePlus Nord 5"
print(f"Searching for: {SEARCH_PRODUCT}")

search_box = driver.find_element(By.ID, 'twotabsearchtextbox')
search_box.clear()
search_box.send_keys(SEARCH_PRODUCT)
time.sleep(1)
search_box.submit()
time.sleep(3)

print("✓ Product search completed!")

Navigating to Amazon India...
No consent dialog found
Searching for: OnePlus Nord 5
✓ Product search completed!


In [4]:
# Section 4: Select First Product and Navigate to Review Page
print("Selecting first product from search results...")

try:
    # Wait for search results
    WebDriverWait(driver, 15).until(
        EC.presence_of_all_elements_located((By.XPATH, "//div[@data-component-type='s-search-result']//h2/a"))
    )
    time.sleep(2)
    
    # Click first product link
    first_product = driver.find_element(By.XPATH, "//div[@data-component-type='s-search-result']//h2/a")
    driver.execute_script("arguments[0].click();", first_product)
    time.sleep(3)
    print("✓ Product page loaded!")
    
except Exception as e:
    print(f"Error selecting product: {str(e)[:100]}")
    print("Attempting to continue...")

# Handle pop-ups
try:
    close_button = driver.find_element(By.CLASS_NAME, 'a-button-close')
    close_button.click()
    time.sleep(1)
except:
    pass

print("Ready to extract reviews!")

Selecting first product from search results...
Error selecting product: Message: 

Attempting to continue...
Ready to extract reviews!


In [6]:
# Section 5: Extract Reviews with Pagination
print("Starting review extraction...\\n")

reviews_data = []
review_count = 0
MIN_REVIEWS = 100
page_count = 0
max_pages = 10

try:
    # First, get the current URL to extract ASIN if needed
    current_url = driver.current_url
    print(f"Current URL: {current_url}")
    
    # Try to navigate to reviews directly via URL structure
    # For product pages, reviews are typically at /reviews/ endpoint
    if '/dp/' in current_url:
        base_url = current_url.split('?')[0]  # Remove query params
        reviews_url = base_url.replace('/dp/', '/reviews/').replace('/product/', '/reviews/')\n        try:\n            print(f"Navigating to reviews page...\")\n            driver.get(reviews_url)\n            time.sleep(3)\n            print(\"✓ Successfully navigated to reviews page\")\n        except:\n            print(\"Could not navigate directly, will extract from current page\")\n    \n    # Extract reviews from each page\n    while review_count < MIN_REVIEWS and page_count < max_pages:\n        page_count += 1\n        print(f\"\\n--- Processing Page {page_count} ---\")\n        \n        # Parse page with BeautifulSoup\n        soup = BeautifulSoup(driver.page_source, 'html.parser')\n        \n        # Try multiple selectors for reviews\n        review_divs = []\n        \n        # Method 1: Standard Amazon data-hook\n        review_divs = soup.find_all('div', {'data-hook': 'review'})\n        \n        # Method 2: Alternative class-based selector\n        if not review_divs:\n            review_divs = soup.find_all('div', {'class': 'a-section review aok-relative'})\n        \n        # Method 3: Generic review container\n        if not review_divs:\n            review_divs = soup.find_all('div', {'id': lambda x: x and 'customer_review' in x})\n        \n        print(f\"Found {len(review_divs)} reviews on this page\")\n        \n        # Extract review information\n        for review_div in review_divs:\n            if review_count >= MIN_REVIEWS:\n                break\n            \n            try:\n                # Extract review text - try multiple selectors\n                review_text_elem = review_div.find('span', {'data-hook': 'review-body'})\n                if not review_text_elem:\n                    review_text_elem = review_div.find('span', string=lambda x: x and len(str(x)) > 20)\n                \n                review_text = review_text_elem.get_text(strip=True) if review_text_elem else None\n                \n                if not review_text or len(review_text) < 10:\n                    continue\n                \n                # Extract rating - try multiple selectors\n                rating_elem = review_div.find('i', {'data-hook': 'review-star-rating'})\n                if not rating_elem:\n                    rating_span = review_div.find('span', {'class': 'a-icon-star'})\n                    if rating_span:\n                        rating_elem = rating_span.find('i')\n                \n                rating_text = rating_elem.get_text(strip=True) if rating_elem else \"N/A\"\n                rating = rating_text.split()[0] if rating_text and rating_text != \"N/A\" else \"N/A\"\n                \n                # Extract reviewer name\n                reviewer_elem = review_div.find('a', {'data-hook': 'review-author'})\n                reviewer_name = reviewer_elem.get_text(strip=True) if reviewer_elem else \"Anonymous\"\n                \n                # Extract review date\n                date_elem = review_div.find('span', {'data-hook': 'review-date'})\n                review_date = date_elem.get_text(strip=True) if date_elem else \"N/A\"\n                \n                # Add to collection\n                reviews_data.append({\n                    'review_text': review_text,\n                    'rating': rating,\n                    'reviewer_name': reviewer_name,\n                    'review_date': review_date\n                })\n                review_count += 1\n                \n            except Exception as e:\n                continue\n        \n        print(f\"Total reviews collected: {review_count}/{MIN_REVIEWS}\")\n        \n        if review_count >= MIN_REVIEWS:\n            print(f\"\\n✓ Reached minimum of {MIN_REVIEWS} reviews!\")\n            break\n        \n        # Go to next page\n        try:\n            # Try different pagination selectors\n            next_button = None\n            \n            # Method 1: Standard next button\n            try:\n                next_button = driver.find_element(By.XPATH, \"//a[contains(@aria-label, 'Next page')]\")\n            except:\n                pass\n            \n            # Method 2: Alternative pagination\n            if not next_button:\n                try:\n                    next_button = driver.find_element(By.CSS_SELECTOR, \"a.s-pagination-next\")\n                except:\n                    pass\n            \n            if next_button and 'disabled' not in next_button.get_attribute('class'):\n                driver.execute_script(\"arguments[0].click();\", next_button)\n                time.sleep(3)\n            else:\n                print(\"No more pages available\")\n                break\n        except Exception as e:\n            print(f\"Could not navigate to next page: {str(e)[:50]}\")\n            break\n    \n    print(f\"\\n{'='*50}\")\n    print(f\"Total reviews collected: {review_count}\")\n    print(f\"{'='*50}\")\n    \nexcept Exception as e:\n    print(f\"Error during extraction: {str(e)[:100]}\")\n    import traceback\n    traceback.print_exc()\n\nfinally:\n    driver.quit()\n    print(\"\\n✓ WebDriver closed!\")

SyntaxError: unexpected character after line continuation character (4206365251.py, line 19)

In [ ]:
# Section 6: Create DataFrame and Save to CSV
print("Creating DataFrame and saving to CSV...\n")

df = pd.DataFrame(reviews_data)

if len(df) > 0:
    print(f"DataFrame created with {len(df)} reviews")
    print(f"\nDataFrame Info:")
    print(df.info())
    print(f"\nFirst few rows:")
    print(df.head(10))
    
    # Save to CSV
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_filename = f"amazon_reviews_{SEARCH_PRODUCT.replace(' ', '_')}_{timestamp}.csv"
    df.to_csv(csv_filename, index=False, encoding='utf-8')
    print(f"\n✓ CSV file saved: {csv_filename}")
else:
    print("No reviews collected!")

In [ ]:
# Section 7: Data Cleaning and Validation
if len(df) > 0:
    print("Validating and cleaning data...\n")
    
    print(f"Original data shape: {df.shape}")
    
    # Remove duplicates
    df_cleaned = df.drop_duplicates(subset=['review_text'])
    print(f"Duplicates removed: {len(df) - len(df_cleaned)} rows")
    
    # Remove null values
    df_cleaned = df_cleaned.dropna(subset=['review_text'])
    
    # Remove HTML tags and extra whitespace
    df_cleaned['review_text'] = df_cleaned['review_text'].str.replace(r'<[^>]+>', '', regex=True)
    df_cleaned['review_text'] = df_cleaned['review_text'].str.replace(r'\s+', ' ', regex=True).str.strip()
    
    print(f"Final cleaned data shape: {df_cleaned.shape}\n")
    
    # Display statistics
    print(f"{'='*50}")
    print("DATA VALIDATION SUMMARY")
    print(f"{'='*50}")
    print(f"Total reviews: {len(df_cleaned)}")
    print(f"\nNull values per column:")
    print(df_cleaned.isnull().sum())
    print(f"\nRating distribution:")
    print(df_cleaned['rating'].value_counts())
    print(f"\nReview text length statistics:")
    print(df_cleaned['review_text'].str.len().describe())
    
    # Save cleaned CSV
    cleaned_filename = csv_filename.replace('.csv', '_cleaned.csv')
    df_cleaned.to_csv(cleaned_filename, index=False, encoding='utf-8')
    print(f"\n✓ Cleaned CSV file saved: {cleaned_filename}")
    
    # Display sample reviews
    print(f"\n{'='*50}")
    print("SAMPLE REVIEWS (First 3)")
    print(f"{'='*50}")
    for idx, row in df_cleaned.head(3).iterrows():
        print(f"\nReview {idx+1}:")
        print(f"Rating: {row['rating']}")
        print(f"Reviewer: {row['reviewer_name']}")
        print(f"Date: {row['review_date']}")
        print(f"Text: {row['review_text'][:200]}...")
        print("-" * 50)
else:
    print("No data to clean!")